# 01. Randomness와 entropy 기초

목표: bit stream의 bias, serial correlation과 보수적인 min-entropy 근사치를 계산한다. 이 notebook은 QRNG를 구현하거나 NIST SP 800-90B validation을 재현하지 않는다. 재현 가능한 교육을 위해 deterministic Python PRNG를 사용한다.

## 1. 세 가지 toy source

균등해 보이는 source, 0이 더 자주 나오는 biased source, 직전 bit를 자주 반복하는 correlated source를 만든다. 실제 entropy source 평가는 physical noise model과 raw sample이 필요하다.

In [ ]:
import math
import random


def iid_bits(rng: random.Random, n: int, probability_one: float = 0.5) -> list[int]:
    return [int(rng.random() < probability_one) for _ in range(n)]


def correlated_bits(rng: random.Random, n: int, repeat_probability: float = 0.9) -> list[int]:
    bits = [rng.randrange(2)]
    for _ in range(n - 1):
        bits.append(bits[-1] if rng.random() < repeat_probability else 1 - bits[-1])
    return bits


rng = random.Random(20260827)
sources = {
    "balanced": iid_bits(rng, 20_000, 0.5),
    "biased": iid_bits(rng, 20_000, 0.2),
    "correlated": correlated_bits(rng, 20_000, 0.9),
}
assert all(len(bits) == 20_000 for bits in sources.values())

## 2. Bias와 lag-1 correlation

평균은 1의 비율이다. Lag-1 correlation은 인접 bit 관계를 보지만 더 긴 memory나 adversarial predictability를 모두 포착하지는 않는다.

In [ ]:
def proportion_one(bits: list[int]) -> float:
    return sum(bits) / len(bits)


def lag1_correlation(bits: list[int]) -> float:
    mean = proportion_one(bits)
    variance = sum((bit - mean) ** 2 for bit in bits)
    if variance == 0:
        return 1.0
    covariance = sum((a - mean) * (b - mean) for a, b in zip(bits, bits[1:]))
    return covariance / variance


for name, bits in sources.items():
    print(name, {"p(1)": round(proportion_one(bits), 4), "lag1": round(lag1_correlation(bits), 4)})

assert proportion_one(sources["biased"]) < 0.25
assert lag1_correlation(sources["correlated"]) > 0.7

## 3. 단순 symbol-frequency min-entropy

`-log2(max(p0,p1))`는 한 symbol의 bias만 보는 교육용 값이다. Correlated source는 0과 1의 빈도가 비슷해도 다음 bit가 예측 가능하므로 이 값이 entropy를 과대평가할 수 있다.

In [ ]:
def frequency_min_entropy(bits: list[int]) -> float:
    p1 = proportion_one(bits)
    return -math.log2(max(p1, 1 - p1))


for name, bits in sources.items():
    print(name, "frequency H_min/bit =", round(frequency_min_entropy(bits), 4))

assert frequency_min_entropy(sources["biased"]) < 0.5
assert frequency_min_entropy(sources["correlated"]) > 0.9

## 4. 해석

Correlated source가 frequency test를 통과해도 안전하지 않다는 것이 핵심이다. 실제 QRNG 평가에서는 noise source의 물리 model, non-IID estimator, startup·continuous health test, conditioning과 failure response를 함께 검증한다. Statistical test 통과만으로 quantum origin이나 cryptographic unpredictability를 증명할 수 없다.